In [0]:
%run ../init_framework_gold

In [0]:
# Get Batch_Id
batch_id = get_batch_id()

In [0]:
# --- Catalog ---
dbutils.widgets.text("catalog_name", "lending_risk_dev") # Default value for manual testing
CATALOG = dbutils.widgets.get("catalog_name")

print(f"Running setup for Catalog: {CATALOG}")

In [0]:

from pyspark.sql.functions import lit

# --- 1. SOURCE INGESTION (Gold Layer) ---

# Read Loans Silver Stream
df_loans = read_delta_stream(
    spark=spark, 
    table_name=LOANS_SILVER
).withColumn("_batch_id", lit(batch_id))

# Read Repayments Silver Stream
df_customers = read_delta_stream(
    spark=spark, 
    table_name=CUSTOMERS_SILVER
).drop("_batch_id")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

# Configured for 8-core cluster (16 shuffle partitions).
spark.conf.set("spark.sql.shuffle.partitions", "16")

# --- 2. STREAMING JOIN & SELECTION ---
# We use aliases 'p' for repayments and 'c' for loans (customers/contracts)
df_joined = df_customers.alias("c").join(
    df_loans.alias("l"),
    "member_id",
    "inner"
).select(
    col("c.member_id"), 
    col("l._batch_id"),
    col("l.loan_status"),
    col("c.home_ownership"),
    col("l.funded_amount"),
    col("c.total_high_credit_limit"),        
    col("c.sub_grade"),    
    col("c.grade"),    
    col("c._change_type").alias("customers_change_type"), # Carried for CDF filtering in 
    col("l._change_type").alias("loans_change_type") # Carried for CDF filtering in     
)

In [0]:
# --- 3. APPLY GOLD METADATA ---
# # Injecting lineage and timestamp
df_with_metadata = apply_gold_metadata(df_joined, lit("calc_financial_performance"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# --- 4. SINK LOGIC (foreachBatch) ---

def upsert_payment_performance_to_gold(microBatchDF, batchId):
    """
    Stateless foreachBatch implementation for Gold Layer.
    Handles CDF metadata filtering, deduplication, scoring via lookup, and Delta Merge.
    """
    spark_session = microBatchDF.sparkSession

    # --- 1. Filter for 'Latest Truth' ---
    # Removes pre-images and accounts for State Store NULLs in the join
    df_clean = microBatchDF.filter(
        (col("customers_change_type").isin("insert", "update_postimage"))
        & (
            col("loans_change_type").isin("insert", "update_postimage")
            | col("customers_change_type").isNull()
        )
    )

    # --- INJECTED: Micro-batch Deduplication ---
    # Partition by member_id, sort by newest update first to keep only the absolute latest state
    window_spec = Window.partitionBy("member_id").orderBy(col("_updated_at").desc())
    
    df_deduplicated = (df_clean
                       .withColumn("rn", row_number().over(window_spec))
                       .filter(col("rn") == 1)
                       .drop("rn"))

    # Register Deduplicated Source View for SQL transformations
    df_deduplicated.createOrReplaceTempView("batch_updates")

    # --- 2. Define Scoring Logic (Categorization) ---
    scoring_query = f""" 
        SELECT 
            member_id,
            _batch_id,
            
            -- 1. Loan Status Category
            CASE 
                WHEN LOWER(loan_status) LIKE '%fully paid%' THEN 'EXCELLENT'
                WHEN LOWER(loan_status) LIKE '%current%' THEN 'GOOD'
                WHEN LOWER(loan_status) LIKE '%in grace period%' THEN 'BAD'
                WHEN LOWER(loan_status) LIKE '%late (16-30 days)%' OR LOWER(loan_status) LIKE '%late (31-120 days)%' THEN 'VERY_BAD'
                WHEN LOWER(loan_status) LIKE '%charged off%' THEN 'UNACCEPTABLE'
                ELSE 'UNACCEPTABLE'
            END AS loan_status_key,
            
            -- 2. Home Ownership Category
            CASE 
                WHEN LOWER(home_ownership) LIKE '%own' THEN 'EXCELLENT'
                WHEN LOWER(home_ownership) LIKE '%rent' THEN 'GOOD'
                WHEN LOWER(home_ownership) LIKE '%mortgage' THEN 'BAD'
                WHEN LOWER(home_ownership) LIKE '%any' OR home_ownership IS NULL THEN 'VERY_BAD'
                ELSE 'UNACCEPTABLE'
            END AS home_ownership_key,
            
            -- 3. Credit Limit Utilization Category
            CASE 
                WHEN funded_amount <= (total_high_credit_limit * 0.10) THEN 'EXCELLENT'
                WHEN funded_amount > (total_high_credit_limit * 0.10) AND funded_amount <= (total_high_credit_limit * 0.20) THEN 'VERY_GOOD'
                WHEN funded_amount > (total_high_credit_limit * 0.20) AND funded_amount <= (total_high_credit_limit * 0.30) THEN 'GOOD'
                WHEN funded_amount > (total_high_credit_limit * 0.30) AND funded_amount <= (total_high_credit_limit * 0.50) THEN 'BAD'
                WHEN funded_amount > (total_high_credit_limit * 0.50) AND funded_amount <= (total_high_credit_limit * 0.70) THEN 'VERY_BAD'
                WHEN funded_amount > (total_high_credit_limit * 0.70) THEN 'UNACCEPTABLE'
                ELSE 'UNACCEPTABLE'
            END AS credit_limit_key,
            
            -- 4a. Base Grade Category
            CASE 
                WHEN grade = 'A' THEN 'EXCELLENT'
                WHEN grade = 'B' THEN 'VERY_GOOD'
                WHEN grade = 'C' THEN 'GOOD'
                WHEN grade = 'D' THEN 'BAD'
                WHEN grade = 'E' THEN 'VERY_BAD'
                WHEN grade IN ('F', 'G') THEN 'UNACCEPTABLE'
                ELSE 'UNACCEPTABLE'
            END AS grade_key,
            
            -- 4b. Sub-Grade Multiplier (Extracted for downstream math)
            CASE 
                WHEN sub_grade IN ('A1', 'B1', 'C1', 'D1', 'E1') THEN 1.00
                WHEN sub_grade IN ('A2', 'B2', 'C2', 'D2', 'E2') THEN 0.95
                WHEN sub_grade IN ('A3', 'B3', 'C3', 'D3', 'E3') THEN 0.90
                WHEN sub_grade IN ('A4', 'B4', 'C4', 'D4', 'E4') THEN 0.85
                WHEN sub_grade IN ('A5', 'B5', 'C5', 'D5', 'E5') THEN 0.80
                ELSE 1.00
            END AS grade_multiplier,
            
            -- Metadata Pass-Through for Gold Merge
            _updated_at,
            _scoring_module
        FROM batch_updates
    """

    # Map Keys to Points using Silver Lookup Table
    df_scored = spark_session.sql(scoring_query)
    df_scored.createOrReplaceTempView("scored_updates")

    # --- 3. Points Mapping Logic via Broadcast Join ---
    points_query = f"""
        SELECT 
            perf.member_id,
            perf._batch_id,
            r1.points AS loan_status_pts,
            r2.points AS home_pts,
            r3.points AS credit_limit_pts,
            -- Apply the extracted multiplier and ensure strict INT casting for Delta
            CAST(ROUND(r4.points * perf.grade_multiplier, 0) AS INT) AS grade_pts, 
            perf._updated_at,
            perf._scoring_module
        FROM scored_updates perf 
        JOIN {CATALOG}.silver.ref_score_points_mapping r1
            ON perf.loan_status_key = r1.rating_key
        JOIN {CATALOG}.silver.ref_score_points_mapping r2
            ON perf.home_ownership_key = r2.rating_key
        JOIN {CATALOG}.silver.ref_score_points_mapping r3
            ON perf.credit_limit_key = r3.rating_key
        JOIN {CATALOG}.silver.ref_score_points_mapping r4
            ON perf.grade_key = r4.rating_key
    """

    # Final Preparation for Merge
    df_scored = spark_session.sql(points_query)

    # --- 4. Execute Idempotent Merge into Gold ---
    # Register the mapped integer data as a Temp View
    df_scored.createOrReplaceTempView("financials_batch_updates")

    merge_query = f"""
        MERGE INTO {CATALOG}.gold.calc_financial_performance AS target
        USING financials_batch_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED THEN
            UPDATE SET
                target.loan_status_pts = source.loan_status_pts,
                target.home_pts = source.home_pts,
                target.credit_limit_pts = source.credit_limit_pts,
                target.grade_pts = source.grade_pts,
                target._updated_at = source._updated_at,
                target._scoring_module = source._scoring_module,
                target._batch_id = source._batch_id
        WHEN NOT MATCHED THEN
            INSERT (
                member_id, 
                loan_status_pts, 
                home_pts, 
                credit_limit_pts, 
                grade_pts, 
                _updated_at, 
                _scoring_module,
                _batch_id
            )
            VALUES (
                source.member_id, 
                source.loan_status_pts, 
                source.home_pts, 
                source.credit_limit_pts, 
                source.grade_pts, 
                source._updated_at, 
                source._scoring_module,
                source._batch_id
            )
    """
    
    # Commit to Delta
    spark_session.sql(merge_query)


# --- 5. STREAM TRIGGER ---
# Using availableNow=True for batch-like cost efficiency on a streaming engine
query = (
    df_with_metadata.writeStream
    .outputMode("append")
    .option("checkpointLocation", GOLD_CHECKPOINT_FINANCIAL_PERF)
    .trigger(availableNow=True)
    .foreachBatch(upsert_payment_performance_to_gold)
    .start()
)

# Block notebook termination until micro-batch is committed
query.awaitTermination()

In [0]:
# # Forcefully clear the checkpoint directory to reset stream state
# dbutils.fs.rm(GOLD_CHECKPOINT_FINANCIAL_PERF, recurse=True)

# print(f"Purged: {GOLD_CHECKPOINT_FINANCIAL_PERF}")

In [0]:
spark.sql(f"select * from {CATALOG}.gold.calc_financial_performance limit 3;")